In [10]:
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Load data
titanic = pd.DataFrame(sns.load_dataset('titanic'))
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [11]:
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Load data
titanic = sns.load_dataset('titanic')

# 2. Split into “known‐age” and “missing‐age”
known_age   = titanic[titanic['age'].notnull()].copy()
missing_age = titanic[titanic['age'].isnull()].copy()

# 3. Decide which predictors to use
#    (drop passenger identifiers & the target)
FEATURES = ['pclass','sex','sibsp','parch','fare','embarked','class','who','adult_male','alone']

X_train = known_age[FEATURES]
y_train = known_age['age']
X_pred  = missing_age[FEATURES]

# 4. Build a preprocessing + model pipeline
#    - OneHotEncode categorical columns
#    - Pass through numerical columns
categorical_cols = ['sex','embarked','class','who','adult_male','alone']
numeric_cols     = ['pclass','sibsp','parch','fare']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', 'passthrough', numeric_cols),
])

model = Pipeline([
    ('preproc', preprocessor),
    ('reg', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 5. Train on known ages
model.fit(X_train, y_train)

# 6. Predict the missing ages
predicted_ages = model.predict(X_pred)

# 7. Fill them back into the original DataFrame
titanic.loc[titanic['age'].isnull(), 'age'] = predicted_ages

# 8. (Optional) Check that there are no more nulls
print("Missing ages after imputation:", titanic['age'].isnull().sum())
print(titanic['age'].tail(20))


Missing ages after imputation: 0
871    47.000000
872    33.000000
873    47.000000
874    28.000000
875    15.000000
876    20.000000
877    19.000000
878    27.024868
879    56.000000
880    25.000000
881    33.000000
882    22.000000
883    28.000000
884    25.000000
885    39.000000
886    27.000000
887    19.000000
888    30.840000
889    26.000000
890    32.000000
Name: age, dtype: float64
